Step 0 - Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

Step 1: Webscraping 

In [ ]:
def extraer_partidos_mundial(url):
    """
    Función genérica para extraer tablas de resultados de una URL.
    Idealmente, apuntarías a páginas de resultados históricos (ej. Wikipedia del Mundial).
    """
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Buscar todas las tablas estadísticas (depende de la estructura del sitio)
    tablas = soup.find_all('table', {'class': 'wikitable'}) 
    
    # Usar pandas para leer la tabla HTML directamente
    if tablas:
        # Extraemos la primera tabla encontrada como ejemplo
        df = pd.read_html(str(tablas[0]))[0]
        return df
    return None

# Ejemplo de uso (URL ficticia, debes reemplazarla por una válida con la estructura deseada)
# df_raw = extraer_partidos_mundial('https://en.wikipedia.org/wiki/2022_FIFA_World_Cup')

Step 2: Transform Data

In [ ]:
# Simulamos un dataset extraído crudo para el ejercicio
data = {
    'Local': ['Argentina', 'Brasil', 'Francia'],
    'Visitante': ['Croacia', 'Suiza', 'Marruecos'],
    'Resultado': ['3 - 0', '1 - 0', '2 - 0']
}
df = pd.DataFrame(data)

# 1. Limpieza y extracción de Goles
# Separar el string "3 - 0" en columnas numéricas
df[['Goles_Local', 'Goles_Visitante']] = df['Resultado'].str.split('-', expand=True).astype(int)

# 2. Reestructuración (Melt/Stack) para el Modelo Poisson
# Necesitamos: Equipo, Oponente, Goles, y si jugó de Local (Home Advantage)

partidos_local = df[['Local', 'Visitante', 'Goles_Local']].copy()
partidos_local.columns = ['Equipo', 'Oponente', 'Goles']
partidos_local['Es_Local'] = 1

partidos_visitante = df[['Visitante', 'Local', 'Goles_Visitante']].copy()
partidos_visitante.columns = ['Equipo', 'Oponente', 'Goles']
partidos_visitante['Es_Local'] = 0

# Unimos ambos DataFrames
df_modelo = pd.concat([partidos_local, partidos_visitante], ignore_index=True)

# 3. Encoding (One-Hot Encoding automático en statsmodels)
# Al usar statsmodels, las variables categóricas (Equipo y Oponente) se 
# codifican automáticamente internamente, capturando la "Fuerza Ofensiva" (Equipo)
# y la "Fuerza Defensiva" (Oponente).

print(df_modelo.head())

Step 3: Predicción

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Entrenar el modelo de Poisson
# La fórmula predice Goles basándose en qué Equipo ataca, qué Oponente defiende y si es Local.
formula = 'Goles ~ Equipo + Oponente + Es_Local'

# Ajustamos el modelo (Generalized Linear Model)
modelo_poisson = smf.glm(formula=formula, data=df_modelo, 
                        family=sm.families.Poisson()).fit()

print(modelo_poisson.summary())

Step 4: Simulación y predicción particular

In [ ]:
from scipy.stats import poisson

def predecir_partido(modelo, equipo_local, equipo_visitante):
    # Calcular los goles esperados (Lambda) para el equipo local
    lambda_local = modelo.predict(pd.DataFrame(data={'Equipo': equipo_local, 
                                                    'Oponente': equipo_visitante, 
                                                    'Es_Local': 1})).values[0]
    
    # Calcular los goles esperados (Lambda) para el equipo visitante
    lambda_visitante = modelo.predict(pd.DataFrame(data={'Equipo': equipo_visitante, 
                                                        'Oponente': equipo_local, 
                                                        'Es_Local': 0})).values[0]
    
    # Simular la matriz de resultados posibles (ej. hasta 5 goles)
    max_goles = 5
    probabilidades = np.zeros((max_goles+1, max_goles+1))
    
    for goles_l in range(max_goles+1):
        for goles_v in range(max_goles+1):
            # La probabilidad de un resultado exacto es el producto de las probabilidades individuales
            probabilidades[goles_l, goles_v] = poisson.pmf(goles_l, lambda_local) * poisson.pmf(goles_v, lambda_visitante)
            
    # Calcular probabilidad de Victoria Local, Empate y Victoria Visitante
    prob_victoria_local = np.sum(np.tril(probabilidades, -1))
    prob_empate = np.sum(np.diag(probabilidades))
    prob_victoria_visit = np.sum(np.triu(probabilidades, 1))
    
    return {
        'Goles_Esperados_Local': round(lambda_local, 2),
        'Goles_Esperados_Visitante': round(lambda_visitante, 2),
        'Prob_Victoria_Local': round(prob_victoria_local * 100, 2),
        'Prob_Empate': round(prob_empate * 100, 2),
        'Prob_Victoria_Visitante': round(prob_victoria_visit * 100, 2)
    }

# Ejemplo de predicción
# prediccion = predecir_partido(modelo_poisson, 'Argentina', 'Francia')
# print(prediccion)